In [ ]:
# Cell 1: Install Required Packages

!pip install ASE
!pip install mace-torch ase rdkit weas-widget

In [ ]:
# Cell 2: Import Required Libraries

# General Libraries
import numpy as np                              # Stores numbers in arrays and runs fast calculations on them
import matplotlib.pyplot as plt                 # Plot Graphs
import pandas as pd                             # Opens the data file as a table and find rows by their SMILES string

# Building Geometry of Molecules / Atoms
from rdkit import Chem                          # Used to build molecules (basic package)
from rdkit.Chem import AllChem                  # Used to build molecules (advanced package)

# MACE Library
from mace.calculators import mace_off           # MACE-OFF (Machine Learning Potential)

# Atomic Simulation Environment Libraries
from ase import Atoms                           # Represents a molecule object with information
from ase.build import molecule                  # Creates an atomic structure from the database
from ase.optimize import QuasiNewton            # Optimization / energy minimization
from ase.vibrations import Vibrations           # Used to calculate vibrational modes of the Atom object
from ase.thermochemistry import IdealGasThermo  # Allows you to calculate entropy, enthalpy, and gibbs free energy
from ase.units import kJ, mol                   # Conversion for units

In [ ]:
# Cell 3: Load MACE-OFF

print("Loading MACE-OFF (medium model)...")
calc_mol = mace_off(model="small", default_dtype="float64")
print("MACE-OFF loaded.")

In [ ]:
# Cell 4: Define Function to Compute Chemical Properties

# Geometry, Symmetry, and Spin are used to get accurate properties
def compute_thermo(
    atoms,                      # Attach atoms / molecule
    calc,                       # Attach calculator (MACE-OFF)
    geometry,                   # (Linear or nonlinear)
    symmetrynumber,             # Symmetry of molecule
    spin,                       # 0.5 for each unpaired electron
    temperature=298.15,
    vib_name='molecule_vib',
    fmax=0.01,                  # Force threshold
    vib_energy_threshold=0.01,
    verbose=False
):
    """
    Compute potential energy and enthalpy for a molecule.

    Parameters
    ----------
    atoms : ase.Atoms
        The molecule to compute thermodynamics for.
    calc : ASE calculator
        The calculator to use (e.g. EMT, GPAW, etc.).
    geometry : str
        'linear' or 'nonlinear' — molecular geometry type.
    symmetrynumber : int
        Rotational symmetry number (e.g. 3 for NH3, 2 for H2).
    spin : float
        Total spin (0 for closed-shell, 0.5 per unpaired electron).
    temperature : float
        Temperature in Kelvin. Default: 298.15.
    vib_name : str
        Prefix for vibrational frequency cache files.
    fmax : float
        Force convergence criterion for geometry optimization (eV/Å).
    vib_energy_threshold : float
        Minimum vibrational energy (eV) to include; filters near-zero/imaginary modes.
    verbose : bool
        Whether to print thermochemistry details.

    Returns
    -------
    dict with keys:
        'potential_energy'  : float, eV
        'enthalpy_eV'       : float, eV
        'enthalpy_kJ_mol'   : float, kJ/mol
    """
    # Attach Calculator and Optimize Geometry
    atoms.calc = calc
    dyn = QuasiNewton(atoms, logfile=None)
    dyn.run(fmax=fmax)

    potential_energy = atoms.get_potential_energy()

    # Compute Vibrational Frequencies
    vib = Vibrations(atoms, name=vib_name)
    vib.clean()
    vib.run()
    vib_energies = vib.get_energies()

    # Filter Out Imaginary and Near-zero Modes
    vib_energies = np.array([
        e.real for e in vib_energies
        if e.real > vib_energy_threshold
    ])

    # Compute Chemical Properties
    thermo = IdealGasThermo(
        vib_energies=vib_energies,
        potentialenergy=potential_energy,
        atoms=atoms,
        geometry=geometry,
        symmetrynumber=symmetrynumber,
        spin=spin,
    )

    enthalpy_eV = thermo.get_enthalpy(temperature=temperature, verbose=verbose)
    enthalpy_kJ_mol = enthalpy_eV * (1 / (kJ / mol))

    return {
        'potential_energy': potential_energy,
        'enthalpy_eV': enthalpy_eV,
        'enthalpy_kJ_mol': enthalpy_kJ_mol,
    }

In [ ]:
# Cell 5: Compute Chemical Properties for Ammonia (NH3)

results = compute_thermo(
    atoms=molecule('NH3'),
    calc=calc_mol,
    geometry='nonlinear',
    symmetrynumber=3,
    spin=0,
    temperature=298.15,
    vib_name='nh3_vib',
)

potentialenergy_NH3 = results['potential_energy']
enthalpy_NH3_eV = results['enthalpy_eV']
enthalpy_NH3_kJ_mol = results['enthalpy_kJ_mol']

print(f"NH3 Potential Energy: {potentialenergy_NH3:.4f} eV = {potentialenergy_NH3 * (1 / (kJ / mol)):.4f} kJ")
print(f"Enthalpy of NH3: {enthalpy_NH3_eV:.4f} eV = ")
print(f"Enthalpy of NH3 at 298 K: {enthalpy_NH3_kJ_mol:.4f} kJ/mol")

In [ ]:
# Cell 6: Compute Chemical Properties for CH3CHOHCH3 (Propanol)
# Create Molecule from SMILES, optimize geometry, and compute properties

# Propanol is not in the G2 database, so we need to create the geometry from the SMILES string and optimize it before computing thermochemistry.

def smiles_to_atoms(smiles, seed=64):
    mol = Chem.MolFromSmiles(smiles)
    mol = Chem.AddHs(mol) # Adds explicit hydrogens to the molecule
    AllChem.EmbedMolecule(mol, randomSeed=64) # Randomly places atoms at correct distances from each other
    AllChem.MMFFOptimizeMolecule(mol) # Optimizes the geometry using the MMFF94 classical force field (Starting Position for MACE-OFF optimization))
    conf = mol.GetConformer()
    symbols = [a.GetSymbol() for a in mol.GetAtoms()]
    positions = conf.GetPositions()
    return Atoms(symbols=symbols, positions=positions)

atoms_iPrOH = smiles_to_atoms('CCCO')

results = compute_thermo(
    atoms=atoms_iPrOH,
    calc=calc_mol,
    geometry='nonlinear',
    symmetrynumber=1,
    spin=0,
    temperature=298.15,
    vib_name='1propanol_vib',
)

potentialenergy_CH3CHOHCH3 = results['potential_energy']
enthalpy_CH3CHOHCH3_eV = results['enthalpy_eV']
enthalpy_CH3CHOHCH3_kJ_mol = results['enthalpy_kJ_mol']

print(f"1-Propanol Potential Energy: {potentialenergy_CH3CHOHCH3:.4f} eV = {potentialenergy_CH3CHOHCH3 * (1 / (kJ / mol)):.4f} kJ")
print(f"Enthalpy of CH3CHOHCH3: {enthalpy_CH3CHOHCH3_eV:.4f} eV")
print(f"Enthalpy of CH3CHOHCH3 at 298 K: {enthalpy_CH3CHOHCH3_kJ_mol:.4f} kJ/mol")

In [ ]:
# Cell 7: Compute Chemical Properties for Methanol (CH3OH)

results = compute_thermo(
    atoms=molecule('CH3OH'),
    calc=calc_mol,
    geometry='nonlinear',
    symmetrynumber=1,
    spin=0,
    temperature=298.15,
    vib_name='ch3oh_vib',
)

potentialenergy_CH3OH = results['potential_energy']
enthalpy_CH3OH_eV = results['enthalpy_eV']
enthalpy_CH3OH_kJ_mol = results['enthalpy_kJ_mol']

print(f"CH3OH Potential Energy: {potentialenergy_CH3OH:.4f} eV = {potentialenergy_CH3OH * (1 / (kJ / mol)):.4f} kJ")
print(f"Enthalpy of CH3OH: {enthalpy_CH3OH_eV:.4f} eV")
print(f"Enthalpy of CH3OH at 298 K: {enthalpy_CH3OH_kJ_mol:.4f} kJ/mol")

In [ ]:
# Cell 8: Compute Chemical Properties for C3H8 (Propane)

results = compute_thermo(
    atoms=molecule('C3H8'),
    calc=calc_mol,
    geometry='nonlinear',
    symmetrynumber=2,
    spin=0,
    temperature=298.15,
    vib_name='c3h8_vib',
)

potentialenergy_C3H8 = results['potential_energy']
enthalpy_C3H8_eV = results['enthalpy_eV']
enthalpy_C3H8_kJ_mol = results['enthalpy_kJ_mol']

print(f"C3H8 Potential Energy: {potentialenergy_C3H8:.4f} eV = {potentialenergy_C3H8 * (1 / (kJ / mol)):.4f} kJ")
print(f"Enthalpy of C3H8: {enthalpy_C3H8_eV:.4f} eV")
print(f"Enthalpy of C3H8 at 298 K: {enthalpy_C3H8_kJ_mol:.4f} kJ/mol")

In [ ]:
# Cell 9: Compute Chemical Properties for C4H4S (Thiophene)

results = compute_thermo(
    atoms=molecule('C4H4S'),
    calc=calc_mol,
    geometry='nonlinear',
    symmetrynumber=2,
    spin=0,
    temperature=298.15,
    vib_name='c4h4s_vib',
)

potentialenergy_C4H4S = results['potential_energy']
enthalpy_C4H4S_eV = results['enthalpy_eV']
enthalpy_C4H4S_kJ_mol = results['enthalpy_kJ_mol']

print(f"C4H4S Potential Energy: {potentialenergy_C4H4S:.4f} eV = {potentialenergy_C4H4S * (1 / (kJ / mol)):.4f} kJ")
print(f"Enthalpy of C4H4S: {enthalpy_C4H4S_eV:.4f} eV")
print(f"Enthalpy of C4H4S at 298 K: {enthalpy_C4H4S_kJ_mol:.4f} kJ/mol")

Compute Molecular Properties for Standard Forms

In [ ]:
# Cell 10: Carbon Properties (Assumed to be Graphite, anchored via experimental sublimation enthalpy using NIST)

H_sub_C_kJ = 716.7  # Experimental enthalpy of sublimation: C(graphite) -> C(g), NIST

c_atom = Atoms('C', positions=[[10, 10, 10]], cell=[20, 20, 20], pbc=False)
c_atom.calc = calc_mol # Attach Calculator (MACE-OFF)
E_C = c_atom.get_potential_energy()

# Convert to kJ/mol and add translational thermal contribution (5/2 RT) for consistency
R = 8.314462618e-3  # kJ/(mol K)
T = 298.15
E_C_kJ = E_C * (1 / (kJ / mol))
H_C_atom_kJ = E_C_kJ + (5/2) * R * T # Thermal correction for C atom (Typically IdealGasThermo handles this)

# Anchor to graphite using the sublimation enthalpy cycle:
# E_C(atom) = H(C, graphite) + H_sub  =>  H(C, graphite) = H(C, atom) - H_sub
H_graphite_kJ = H_C_atom_kJ - H_sub_C_kJ

print(f"C atom potential energy: {E_C:.4f} eV = {E_C * (1 / (kJ / mol)):.4f} kJ")
print(f"Enthalpy of C atom at 298 K: {H_C_atom_kJ:.4f} kJ/mol")
print(f"Enthalpy of C (graphite) at 298 K: {H_graphite_kJ:.4f} kJ/mol")

In [ ]:
# Cell 11: Compute Chemical Properties for H2

results = compute_thermo(
    atoms=molecule('H2'),
    calc=calc_mol,
    geometry='linear',
    symmetrynumber=2,
    spin=0,
    temperature=298.15,
    vib_name='h2_vib',
)

potentialenergy_H2 = results['potential_energy']
enthalpy_H2_eV = results['enthalpy_eV']
enthalpy_H2_kJ_mol = results['enthalpy_kJ_mol']

print(f"H2 Potential Energy: {potentialenergy_H2:.4f} eV = {potentialenergy_H2 * (1 / (kJ / mol)):.4f} kJ")
print(f"Enthalpy of H2: {enthalpy_H2_eV:.4f} eV")
print(f"Enthalpy of H2 at 298 K: {enthalpy_H2_kJ_mol:.4f} kJ/mol")

In [ ]:
# Cell 12: Compute Chemical Properties for N2

results = compute_thermo(
    atoms=molecule('N2'),
    calc=calc_mol,
    geometry='linear',
    symmetrynumber=2,
    spin=0,
    temperature=298.15,
    vib_name='n2_vib',
)

potentialenergy_N2 = results['potential_energy']
enthalpy_N2_eV = results['enthalpy_eV']
enthalpy_N2_kJ_mol = results['enthalpy_kJ_mol']

print(f"N2 Potential Energy: {potentialenergy_N2:.4f} eV = {potentialenergy_N2 * (1 / (kJ / mol)):.4f} kJ")
print(f"Enthalpy of N2: {enthalpy_N2_eV:.4f} eV")
print(f"Enthalpy of N2 at 298 K: {enthalpy_N2_kJ_mol:.4f} kJ/mol")

In [ ]:
# Cell 13: Compute Chemical Properties for O2

results = compute_thermo(
    atoms=molecule('O2'),
    calc=calc_mol,
    geometry='linear',
    symmetrynumber=2,
    spin=1,
    temperature=298.15,
    vib_name='o2_vib',
)

potentialenergy_O2 = results['potential_energy']
enthalpy_O2_eV = results['enthalpy_eV']
enthalpy_O2_kJ_mol = results['enthalpy_kJ_mol']

print(f"O2 Potential Energy: {potentialenergy_O2:.4f} eV = {potentialenergy_O2 * (1 / (kJ / mol)):.4f} kJ")
print(f"Enthalpy of O2: {enthalpy_O2_eV:.4f} eV")
print(f"Enthalpy of O2 at 298 K: {enthalpy_O2_kJ_mol:.4f} kJ/mol")

In [ ]:
# Cell 14: Compute Chemical Properties for S (rhombic)

H_sub_S_kJ = 277.0 / 8  # Experimental enthalpy of sublimation: S(rhombic) -> S(g), NIST

s_atom = Atoms('S', positions=[[10, 10, 10]], cell=[20, 20, 20], pbc=False)
s_atom.calc = calc_mol
E_S = s_atom.get_potential_energy()

E_S_kJ = E_S * (1 / (kJ / mol))
H_S_atom_kJ = E_S_kJ + (5/2) * R * T # Thermal Correction for S atom (Typically IdealGasThermo handles this)
H_rhombic_S_kJ = H_S_atom_kJ - H_sub_S_kJ

print(f"S atom potential energy: {E_S:.4f} eV = {E_S * (1 / (kJ / mol)):.4f} kJ")
print(f"Enthalpy of S atom at 298 K: {H_S_atom_kJ:.4f} kJ/mol")
print(f"Enthalpy of S (rhombic) at 298 K: {H_rhombic_S_kJ:.4f} kJ/mol")

In [ ]:
# Cell 15: Compute Error in MACE-OFF vs Experimental Data

# Ammonia
dH_NH3 = enthalpy_NH3_kJ_mol - (0.5*enthalpy_N2_kJ_mol + 1.5 * enthalpy_H2_kJ_mol)
dH_Act_NH3 = -45.9 # Experimental, kJ/mol

print(f"Enthalpy change for NH3 formation at 298 K: {(dH_NH3):.4f} kJ/mol")
print(f"Experimental: {dH_Act_NH3} kJ/mol")

Percent_Error_NH3_MACE = abs((dH_NH3 - dH_Act_NH3) / dH_Act_NH3) * 100
print(f"Percent Error: {Percent_Error_NH3_MACE:.2f}%")
print("")



# Propanol
dH_Pro = enthalpy_CH3CHOHCH3_kJ_mol - (3*H_graphite_kJ + 4*enthalpy_H2_kJ_mol + 0.5*enthalpy_O2_kJ_mol)
dH_Act_Pro = -256.0 # Experimental, kJ/mol

print(f"Enthalpy change for ISP formation at 298 K: {(dH_Pro):.4f} kJ/mol")
print(f"Experimental: {dH_Act_Pro} kJ/mol")

Percent_Error_Pro_MACE = abs((dH_Pro - dH_Act_Pro) / dH_Act_Pro) * 100
print(f"Percent Error: {Percent_Error_Pro_MACE:.2f}%")
print("")



# Methanol
dH_CH3OH = enthalpy_CH3OH_kJ_mol - (H_graphite_kJ + 2*enthalpy_H2_kJ_mol + 0.5*enthalpy_O2_kJ_mol)

dH_Act_CH3OH = -205.0  # Experimental, kJ/mol

print(f"Enthalpy change for CH3OH formation at 298 K: {dH_CH3OH:.4f} kJ/mol")
print(f"Experimental: {dH_Act_CH3OH} kJ/mol")

Percent_Error_CH3OH_MACE = abs((dH_CH3OH - dH_Act_CH3OH) / dH_Act_CH3OH) * 100
print(f"Percent Error: {Percent_Error_CH3OH_MACE:.2f}%")
print("")



# Propane
dH_C3H8 = enthalpy_C3H8_kJ_mol - (3*H_graphite_kJ + 4*enthalpy_H2_kJ_mol)

dH_Act_C3H8 = -104.7  # Experimental, kJ/mol

print(f"Enthalpy change for C3H8 formation at 298 K: {dH_C3H8:.4f} kJ/mol")
print(f"Experimental: {dH_Act_C3H8} kJ/mol")

Percent_Error_C3H8_MACE = abs((dH_C3H8 - dH_Act_C3H8) / dH_Act_C3H8) * 100
print(f"Percent Error: {Percent_Error_C3H8_MACE:.2f}%")
print("")



# Thiophene
dH_C4H4S = enthalpy_C4H4S_kJ_mol - (4*H_graphite_kJ + 2*enthalpy_H2_kJ_mol + H_rhombic_S_kJ)

dH_Act_C4H4S = 116.4  # Experimental, kJ/mol

print(f"Enthalpy change for C4H4S formation at 298 K: {dH_C4H4S:.4f} kJ/mol")
print(f"Experimental: {dH_Act_C4H4S} kJ/mol")

Percent_Error_C4H4S_MACE = abs((dH_C4H4S - dH_Act_C4H4S) / dH_Act_C4H4S) * 100
print(f"Percent Error: {Percent_Error_C4H4S_MACE:.2f}%")

Obtain Values from Database Method

In [ ]:
# Cell 16: Define Location of Database (Located in Github Repository)

DATABASE_PATH = "https://raw.githubusercontent.com/cacherowan/CACHE-Rowan/main/Reference_Files/Chemical_Property_Database/Processed_Solvent_DF_v6_TEST.xlsx"
df = pd.read_excel(DATABASE_PATH)

In [ ]:
# Cell 17: Define Function to Obtain Values from Database Method

def obtain_Enthalpy_Of_Formation(MOLECULE, SMILES):
    """
    Find Properties of input Molecule using the ANN model and database.

    Parameters:
        MOLECULE (str): Name of the molecule (for display purposes).
        SMILES (str): SMILES string of the molecule to look up in the database.

    Returns:
        float: Predicted standard formation enthalpy in kJ/mol.
    """

    # 2. Read the DB:
    db = pd.read_excel(DATABASE_PATH)
    db = db.set_index('SMILES')
    descriptors = db.loc[SMILES]

    Enthalpy_Of_Formation = ['Standard Formation Enthalpy (Gas) [J/mol]']

    Property = descriptors.loc[Enthalpy_Of_Formation]

    return descriptors['Standard Formation Enthalpy (Gas) [J/mol]']

In [ ]:
# Cell 18: Use Function to Obtain Values of Standard Enthalpy of Formation in (J/mol)

Ammonia_Enthalpy_Of_Formation = obtain_Enthalpy_Of_Formation("Ammonia", "N")
print("Ammonia Enthalpy of Formation", Ammonia_Enthalpy_Of_Formation, "J/mol")

Propanol_Enthalpy_Of_Formation = obtain_Enthalpy_Of_Formation("Propanol", "CCCO")
print("Propanol Enthalpy of Formation", Propanol_Enthalpy_Of_Formation, "J/mol")

Methanol_Enthalpy_Of_Formation = obtain_Enthalpy_Of_Formation("Methanol", "CO")
print("Methanol Enthalpy of Formation", Methanol_Enthalpy_Of_Formation, "J/mol")

Propane_Enthalpy_Of_Formation = obtain_Enthalpy_Of_Formation("Propane", "CCC")
print("Propane Enthalpy of Formation", Propane_Enthalpy_Of_Formation, "J/mol")

Thiophene_Enthalpy_Of_Formation = obtain_Enthalpy_Of_Formation("Thiophene", "s1cccc1")
print("Thiophene Enthalpy of Formation", Thiophene_Enthalpy_Of_Formation, "J/mol")

In [ ]:
# Cell 19: Calculate Percent Error in ANN Model vs Experimental Data (NIST)

# Convert from J/mol to kJ/mol
Ammonia_Enthalpy_Of_Formation = Ammonia_Enthalpy_Of_Formation / 1000
Propanol_Enthalpy_Of_Formation = Propanol_Enthalpy_Of_Formation / 1000
Methanol_Enthalpy_Of_Formation = Methanol_Enthalpy_Of_Formation / 1000
Propane_Enthalpy_Of_Formation = Propane_Enthalpy_Of_Formation / 1000
Thiophene_Enthalpy_Of_Formation = Thiophene_Enthalpy_Of_Formation / 1000

Percent_Error_NH3_ANN = abs((Ammonia_Enthalpy_Of_Formation - -45.9) / -45.9) * 100

Percent_Error_CH3CHOHCH3_ANN = abs((Propanol_Enthalpy_Of_Formation - -256.0) / -256.0) * 100

Percent_Error_CH3OH_ANN = abs((Methanol_Enthalpy_Of_Formation - -205.0) / -205.0) * 100

Percent_Error_C3H8_ANN = abs((Propane_Enthalpy_Of_Formation - -104.7) / -104.7) * 100

Percent_Error_C4H4S_ANN = abs((Thiophene_Enthalpy_Of_Formation - 116.4) / 116.4) * 100

In [ ]:
# Cell 20: Show Table of All Percent Errors

import pandas as pd

data = {
    "Chemical Name": ["Ammonia", "Propanol", "Methanol", "Propane", "Thiophene"],
    "MACE-OFF Standard Formation Enthalpy [kJ/mol]": [dH_NH3, dH_Pro, dH_CH3OH, dH_C3H8, dH_C4H4S],
    "Database Standard Formation Enthalpy [kJ/mol]": [Ammonia_Enthalpy_Of_Formation, Propanol_Enthalpy_Of_Formation, Methanol_Enthalpy_Of_Formation, Propane_Enthalpy_Of_Formation, Thiophene_Enthalpy_Of_Formation],
    "Experimental Standard Formation Enthalpy [kJ/mol]": [-45.9, -256.0, -205, -104.7, 116.4], # Values from NIST in kJ/mol
    "MACE-OFF Percent Error": [Percent_Error_NH3_MACE, Percent_Error_Pro_MACE, Percent_Error_CH3OH_MACE, Percent_Error_C3H8_MACE, Percent_Error_C4H4S_MACE],
    "Database Percent Error": [Percent_Error_NH3_ANN, Percent_Error_CH3CHOHCH3_ANN, Percent_Error_CH3OH_ANN, Percent_Error_C3H8_ANN, Percent_Error_C4H4S_ANN]
}
pd.set_option("display.width", 1000)
df = pd.DataFrame(data)

df['MACE-OFF Percent Error'] = df['MACE-OFF Percent Error'].map(lambda x: f'{x:.3g}%')
df['Database Percent Error'] = df['Database Percent Error'].map(lambda x: f'{x:.3g}%')

display(df)